# Notebook 01: Vector Space Fundamentals, Distance Metrics & Top-K Search

Welcome to the **Vector Search & Indexing from Scratch** series!

In the previous module (`embedding_models/`), you mastered how deep learning models transform raw unstructured text into dense semantic embeddings:
$$\vec{v} \in \mathbb{R}^d$$

Now, we enter the engine room of modern **Retrieval-Augmented Generation (RAG)** and **Vector Databases**: **Vector Search & Indexing**.

---

### The High-Level RAG Architecture
Every production RAG system relies on two critical stages:
1. **Offline Indexing Phase**: Chunks are embedded and stored in a vector index.
2. **Online Query Phase**: A user query is embedded, and the retriever finds the $K$ most semantically relevant vectors.

```
       [User Query]
             │
             ▼
   [Embedding Model]  ───▶  Query Vector q ∈ ℝ^d
                                   │
                                   ▼
                       ┌────────────────────────┐
                       │  Vector Search Engine  │ ◀─── Corpus Vectors X ∈ ℝ^(N×d)
                       │  • Metric (Cos, L2, IP)│
                       │  • Top-K Selection     │
                       └────────────────────────┘
                                   │
                                   ▼
                        Top-K Document Chunks
                                   │
                                   ▼
                         [LLM Context Prompt]
```

### Learning Objectives in this Notebook:
1. **Vector Space Geometry**: Understand how dense vectors behave in high-dimensional Euclidean space $\mathbb{R}^d$ and on the unit hypersphere $\mathbb{S}^{d-1}$.
2. **Distance & Similarity Metrics from Scratch**:
   - Euclidean Distance ($L_2$) and the Squared Euclidean optimization.
   - Manhattan Distance ($L_1$).
   - Dot Product (Inner Product / Maximum Inner Product Search - MIPS).
   - Cosine Similarity & Cosine Distance.
3. **The Fundamental Equivalence**: Mathematically prove and empirically demonstrate why **Cosine Similarity $\equiv$ Dot Product** when vectors are $L_2$-normalized, and how modern vector engines exploit this for $10\times$ speedups.
4. **Metric Selection Rules for RAG**: How to choose the right metric for popular models (OpenAI, Cohere, BGE, ColBERT) and the catastrophic "Metric Mismatch" pitfall.
5. **Top-K Selection Algorithms from Scratch**:
   - Naive Full Sort: $O(N \log N)$
   - Priority Queue / Bounded Min-Heap: $O(N \log K)$
   - QuickSelect (Hoare's Selection): $O(N)$ expected time
6. **Empirical Benchmarking**: Compare scaling characteristics as database size $N$ increases.
7. **Industry Reference Implementations**: Contrast our scratch code with NumPy, SciPy, PyTorch, and Faiss.
8. **Pros, Cons & Decision Matrix**: A complete reference guide for your system designs.


---
## 1. Setup & Environment

Let's begin by importing standard Python data structures and scientific computing libraries.
We will use pure Python and NumPy for our from-scratch implementations, and later reference SciPy and Faiss for industry comparisons.


In [ ]:
import math
import heapq
import time
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")
print("Environment configured successfully!")


---
## 2. Vector Space Geometry in Modern RAG

In RAG, an embedding is simply an ordered list of $d$ real numbers representing the semantic position of a piece of text within a high-dimensional vector space:
$$\mathbf{x} = [x_1, x_2, \dots, x_d]^T \in \mathbb{R}^d$$

Typical embedding dimensions in production:
- **Small models**: $d = 384$ (`all-MiniLM-L6-v2`)
- **Medium models**: $d = 768$ (`bge-base`, `bert-base`, `nomic-embed-text`)
- **Large models**: $d = 1024$ (`bge-large`, `gte-large`)
- **Frontier models**: $d = 1536$ (`text-embedding-3-small`, `ada-002`) or $d = 3072$ (`text-embedding-3-large`)

### Dense vs. Sparse Vectors
- **Dense Vectors**: Every coordinate is a continuous float (typically `float32`). Encodes deep semantic associations, synonyms, and context.
- **Sparse Vectors**: Most coordinates are $0$ (e.g., BM25 or SPLADE, where $d \approx 30,000+$ corresponding to vocabulary tokens). Encodes exact keyword matches.

### Vector Magnitude ($L_2$ Norm) and Normalization
The length (magnitude) of a vector in Euclidean space is given by its $L_2$ norm:
$$\|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^d x_i^2} = \sqrt{\mathbf{x} \cdot \mathbf{x}}$$

A vector is **unit-normalized** when its length is exactly 1:
$$\hat{\mathbf{x}} = \frac{\mathbf{x}}{\|\mathbf{x}\|_2} \implies \|\hat{\mathbf{x}}\|_2 = 1$$

Unit vectors all lie on the surface of a **unit hypersphere** $\mathbb{S}^{d-1}$ centered at the origin.


In [ ]:
# Let's generate a batch of synthetic embeddings to visualize geometry
d = 384  # embedding dimension
N = 5    # number of documents

# Generate random embeddings from standard normal distribution
embeddings = np.random.randn(N, d).astype(np.float32)

print("Shape of embeddings matrix (N x d):", embeddings.shape)

# Compute L2 norm for each vector from scratch
# ||x||_2 = sqrt(sum(x_i^2))
scratch_norms = np.sqrt(np.sum(embeddings ** 2, axis=1, keepdims=True))

# Verify against numpy's linalg.norm
np_norms = np.linalg.norm(embeddings, ord=2, axis=1, keepdims=True)
assert np.allclose(scratch_norms, np_norms), "Norm calculation mismatch!"

print("Computed L2 Norms of raw vectors:")
for i, norm_val in enumerate(scratch_norms.flatten()):
    print(f"  Vector {i}: ||x||_2 = {norm_val:.4f}")

# Normalize vectors to unit length
unit_embeddings = embeddings / scratch_norms
unit_norms = np.linalg.norm(unit_embeddings, axis=1)
print("\nL2 Norms after unit normalization:")
for i, norm_val in enumerate(unit_norms):
    print(f"  Vector {i}: ||x_hat||_2 = {norm_val:.4f}")


---
## 3. Distance and Similarity Metrics (From Scratch)

When a user asks a question, the vector database must compute the semantic affinity between the query vector $\mathbf{q}$ and every candidate document $\mathbf{x}_i$.

There are two primary paradigms:
1. **Distance Metrics**: Smaller values mean closer / more similar ($d(\mathbf{u}, \mathbf{v}) \ge 0$).
2. **Similarity Metrics**: Larger values mean closer / more similar.

Let's implement the four core metrics from scratch, explore their mathematical derivations, and highlight implementation tricks used in production engines.


### 3.1 Euclidean Distance ($L_2$ Distance)

Euclidean distance measures the physical "straight-line" distance between two coordinate points in $\mathbb{R}^d$:
$$d_{L2}(\mathbf{u}, \mathbf{v}) = \sqrt{\sum_{i=1}^d (u_i - v_i)^2} = \|\mathbf{u} - \mathbf{v}\|_2$$

#### The "Squared Euclidean" Production Trick
Notice the square root $\sqrt{\cdot}$. In computer hardware, $\sqrt{z}$ is computationally expensive compared to addition and multiplication.

Because the square root is a **strictly monotonically increasing function** for non-negative numbers:
$$a < b \iff \sqrt{a} < \sqrt{b}$$

Therefore, the **ordering / ranking** of nearest neighbors under squared Euclidean distance is **100% identical** to standard Euclidean distance:
$$d_{L2}^2(\mathbf{u}, \mathbf{v}) = \sum_{i=1}^d (u_i - v_i)^2$$

> **Production Tip**: High-performance vector engines (like Faiss and Annoy) always compute $d_{L2}^2$ during nearest neighbor search and only apply $\sqrt{\cdot}$ if the caller explicitly requests the exact distance value!


In [ ]:
def euclidean_distance_scratch(u: np.ndarray, v: np.ndarray) -> float:
    """Compute standard L2 distance from scratch using pure loops/primitives."""
    total_sq_diff = 0.0
    for i in range(len(u)):
        diff = u[i] - v[i]
        total_sq_diff += diff * diff
    return math.sqrt(total_sq_diff)

def euclidean_distance_vectorized(u: np.ndarray, v: np.ndarray) -> float:
    """Vectorized L2 distance using NumPy."""
    diff = u - v
    return float(np.sqrt(np.dot(diff, diff)))

def squared_euclidean_vectorized(u: np.ndarray, v: np.ndarray) -> float:
    """Production optimization: Squared L2 distance (avoids sqrt)."""
    diff = u - v
    return float(np.dot(diff, diff))

# Verification on toy vectors
u_toy = np.array([1.0, 2.0, 3.0], dtype=np.float32)
v_toy = np.array([4.0, 0.0, -1.0], dtype=np.float32)

d_scratch = euclidean_distance_scratch(u_toy, v_toy)
d_vec = euclidean_distance_vectorized(u_toy, v_toy)
d_sq = squared_euclidean_vectorized(u_toy, v_toy)

print(f"Vector u: {u_toy}")
print(f"Vector v: {v_toy}")
print(f"Scratch L2 Distance:          {d_scratch:.6f}")
print(f"Vectorized L2 Distance:       {d_vec:.6f}")
print(f"Squared L2 Distance (d^2):    {d_sq:.6f}  (Notice: {d_vec:.6f}^2 = {d_vec**2:.6f})")
assert np.isclose(d_scratch, d_vec), "Mismatch in L2 implementations!"
assert np.isclose(d_scratch**2, d_sq), "Mismatch in Squared L2!"
print(" Euclidean implementations match perfectly!")


### 3.2 Manhattan Distance ($L_1$ Distance / Taxicab Distance)

Manhattan distance measures the distance between two points along axis-parallel grid lines (like navigating city blocks in Manhattan):
$$d_{L1}(\mathbf{u}, \mathbf{v}) = \sum_{i=1}^d |u_i - v_i|$$

#### Why does $L_1$ exist in vector search?
- In very high dimensions ($d > 1000$), $L_1$ distance is theoretically less prone to the "distance concentration" problem caused by the curse of dimensionality than $L_2$.
- Robust to extreme outlier values along individual dimensions (since differences are not squared).
- Widely used for sparse embeddings, frequency distributions, and histogram matching.


In [ ]:
def manhattan_distance_scratch(u: np.ndarray, v: np.ndarray) -> float:
    """Compute L1 distance from scratch."""
    total_abs_diff = 0.0
    for i in range(len(u)):
        total_abs_diff += abs(u[i] - v[i])
    return total_abs_diff

def manhattan_distance_vectorized(u: np.ndarray, v: np.ndarray) -> float:
    """Vectorized L1 distance."""
    return float(np.sum(np.abs(u - v)))

d1_scratch = manhattan_distance_scratch(u_toy, v_toy)
d1_vec = manhattan_distance_vectorized(u_toy, v_toy)

print(f"Scratch L1 Distance:    {d1_scratch:.6f}")
print(f"Vectorized L1 Distance: {d1_vec:.6f}")
assert np.isclose(d1_scratch, d1_vec), "Mismatch in L1 implementations!"
print(" Manhattan implementations match!")


### 3.3 Dot Product (Inner Product / MIPS)

The Dot Product (also called Inner Product) is the sum of the element-wise products of two vectors:
$$\langle \mathbf{u}, \mathbf{v} \rangle = \mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^d u_i v_i = \mathbf{u}^T \mathbf{v}$$

Geometrically, the dot product combines **angle** and **magnitude**:
$$\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 \cos(\theta)$$

Where:
- $\theta$ is the angle between $\mathbf{u}$ and $\mathbf{v}$.
- $\|\mathbf{u}\|_2, \|\mathbf{v}\|_2$ are the vector lengths.

#### Maximum Inner Product Search (MIPS)
When similarity is evaluated via dot product, finding the top documents is called **MIPS**:
$$\arg\max_{i} \langle \mathbf{q}, \mathbf{x}_i \rangle$$

- **Range**: $(-\infty, +\infty)$ for unconstrained vectors.
- Higher value = higher similarity.
- **Important**: If vectors have unequal lengths, a longer vector can score higher than a shorter vector even if its semantic angle $\theta$ is wider! This is why unnormalized dot product is favored when model training explicitly encodes document quality, authority, or length into the vector's norm.


In [ ]:
def dot_product_scratch(u: np.ndarray, v: np.ndarray) -> float:
    """Compute dot product using primitive loops."""
    total = 0.0
    for i in range(len(u)):
        total += u[i] * v[i]
    return total

def dot_product_vectorized(u: np.ndarray, v: np.ndarray) -> float:
    """Vectorized dot product."""
    return float(np.dot(u, v))

ip_scratch = dot_product_scratch(u_toy, v_toy)
ip_vec = dot_product_vectorized(u_toy, v_toy)

print(f"Scratch Dot Product:    {ip_scratch:.6f}")
print(f"Vectorized Dot Product: {ip_vec:.6f}")
assert np.isclose(ip_scratch, ip_vec), "Mismatch in Dot Product!"
print(" Dot product implementations match!")


### 3.4 Cosine Similarity & Cosine Distance

Cosine similarity measures the **cosine of the angle $\theta$** between two vectors, completely **ignoring their lengths**:
$$\text{Cosine}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \frac{\sum_{i=1}^d u_i v_i}{\sqrt{\sum u_i^2} \sqrt{\sum v_i^2}} = \cos(\theta)$$

- **Range**: $[-1, +1]$
  - $+1$: Vectors point in the exact same direction (angle $\theta = 0^\circ$).
  - $0$: Vectors are orthogonal / uncorrelated (angle $\theta = 90^\circ$).
  - $-1$: Vectors point in diametrically opposite directions (angle $\theta = 180^\circ$).

#### Cosine Distance
Because vector indices generally operate on *distances* (where $0$ means identical), **Cosine Distance** is defined as:
$$d_{\cos}(\mathbf{u}, \mathbf{v}) = 1 - \text{Cosine}(\mathbf{u}, \mathbf{v})$$
- Range: $[0, 2]$
  - $0$: Identical direction
  - $1$: Orthogonal
  - $2$: Opposite direction

> **Numerical Stability Note**: If a vector has all zeros (norm = 0), dividing by norm produces `NaN`. Production code must guard against this using a small epsilon $\epsilon = 10^{-12}$ or checking for zero.


In [ ]:
def cosine_similarity_scratch(u: np.ndarray, v: np.ndarray, eps: float = 1e-12) -> float:
    """Compute cosine similarity from scratch with numerical stability epsilon."""
    dot = 0.0
    norm_u_sq = 0.0
    norm_v_sq = 0.0
    for i in range(len(u)):
        dot += u[i] * v[i]
        norm_u_sq += u[i] * u[i]
        norm_v_sq += v[i] * v[i]
    
    norm_u = math.sqrt(norm_u_sq)
    norm_v = math.sqrt(norm_v_sq)
    
    denominator = max(norm_u * norm_v, eps)
    return dot / denominator

def cosine_similarity_vectorized(u: np.ndarray, v: np.ndarray, eps: float = 1e-12) -> float:
    """Vectorized cosine similarity."""
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    return float(np.dot(u, v) / max(norm_u * norm_v, eps))

def cosine_distance_vectorized(u: np.ndarray, v: np.ndarray) -> float:
    """Cosine distance: d_cos = 1 - cosine_similarity."""
    return 1.0 - cosine_similarity_vectorized(u, v)

cos_sim = cosine_similarity_vectorized(u_toy, v_toy)
cos_dist = cosine_distance_vectorized(u_toy, v_toy)

print(f"Cosine Similarity: {cos_sim:.6f}")
print(f"Cosine Distance:   {cos_dist:.6f}")
assert np.isclose(cosine_similarity_scratch(u_toy, v_toy), cos_sim), "Mismatch in Cosine!"
print(" Cosine implementations match!")


---
## 4. The Fundamental Mathematical Equivalence

This is **the single most important mathematical insight in vector search engineering**:

> If two vectors $\mathbf{u}$ and $\mathbf{v}$ are unit-normalized (i.e., $\|\mathbf{u}\|_2 = 1$ and $\|\mathbf{v}\|_2 = 1$), then:
> 1. **Cosine Similarity is identically equal to the Dot Product.**
> 2. **Squared Euclidean Distance is directly proportional to (1 - Cosine Similarity).**

### Formal Proof:
Let $\mathbf{u}, \mathbf{v} \in \mathbb{R}^d$ such that $\|\mathbf{u}\|_2 = 1$ and $\|\mathbf{v}\|_2 = 1$.

#### Step 1: Expand the squared Euclidean distance:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = \sum_{i=1}^d (u_i - v_i)^2$$
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = \sum_{i=1}^d u_i^2 + \sum_{i=1}^d v_i^2 - 2 \sum_{i=1}^d u_i v_i$$
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = \|\mathbf{u}\|_2^2 + \|\mathbf{v}\|_2^2 - 2(\mathbf{u} \cdot \mathbf{v})$$

#### Step 2: Substitute $\|\mathbf{u}\|_2^2 = 1$ and $\|\mathbf{v}\|_2^2 = 1$:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 1 + 1 - 2(\mathbf{u} \cdot \mathbf{v}) = 2 - 2(\mathbf{u} \cdot \mathbf{v})$$
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 2(1 - \mathbf{u} \cdot \mathbf{v})$$

#### Step 3: Relate to Cosine Similarity:
Since $\|\mathbf{u}\| = \|\mathbf{v}\| = 1$, by definition:
$$\text{Cosine}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{1 \cdot 1} = \mathbf{u} \cdot \mathbf{v}$$

Substituting $\text{Cosine}(\mathbf{u}, \mathbf{v})$ into Step 2:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 2(1 - \text{Cosine}(\mathbf{u}, \mathbf{v})) = 2 \cdot d_{\cos}(\mathbf{u}, \mathbf{v})$$

Rearranging for Cosine Similarity:
$$\mathbf{u} \cdot \mathbf{v} = \text{Cosine}(\mathbf{u}, \mathbf{v}) = 1 - \frac{1}{2} \|\mathbf{u} - \mathbf{v}\|_2^2$$

$$\blacksquare$$

---

### Why is this theorem a superpower in RAG Engineering?

| Metric | Without Normalization | With Pre-Normalization ($L_2$) |
| :--- | :--- | :--- |
| **Cosine Similarity** | Requires computing dot product AND two vector norms at query time: $\frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$ | **Becomes a simple Dot Product**: $\mathbf{u} \cdot \mathbf{v}$ |
| **Computational Cost** | High (divisions, square roots for every candidate) | Extremely Low (single BLAS `gemm` matrix multiply) |
| **Index Flexibility** | Needs specialized Cosine metric | Can use an $L_2$ index OR an Inner Product index! |

Let's test this equivalence empirically across a dataset of 1,000 vectors!


In [ ]:
# Generate 1,000 random document vectors and 1 query vector in 128 dimensions
dim = 128
num_docs = 1000

corpus_raw = np.random.randn(num_docs, dim).astype(np.float32)
query_raw = np.random.randn(dim).astype(np.float32)

# Unit normalize query and corpus
corpus_norm = corpus_raw / np.linalg.norm(corpus_raw, axis=1, keepdims=True)
query_norm = query_raw / np.linalg.norm(query_raw)

# 1. Compute Cosine Similarities
cosine_sims = np.dot(corpus_norm, query_norm) / (
    np.linalg.norm(corpus_norm, axis=1) * np.linalg.norm(query_norm)
)

# 2. Compute Dot Products on normalized vectors
dot_products = np.dot(corpus_norm, query_norm)

# 3. Compute Squared L2 distances
squared_l2_dists = np.sum((corpus_norm - query_norm) ** 2, axis=1)

# Check the exact mathematical identity: Dot Product == 1 - 0.5 * Squared_L2
predicted_dot = 1.0 - 0.5 * squared_l2_dists

# Verify differences are zero up to floating point precision
max_diff_cos_dot = np.max(np.abs(cosine_sims - dot_products))
max_diff_formula = np.max(np.abs(dot_products - predicted_dot))

print(f"Max difference between Cosine Sim and Normalized Dot Product: {max_diff_cos_dot:.2e}")
print(f"Max difference between Dot Product and (1 - 0.5 * L2^2):       {max_diff_formula:.2e}")

# Check ranking equivalence:
# Highest Cosine Sim <=> Highest Dot Product <=> Lowest L2 Distance
rank_by_cosine = np.argsort(-cosine_sims)[:10]       # Top-10 largest
rank_by_dot    = np.argsort(-dot_products)[:10]      # Top-10 largest
rank_by_l2     = np.argsort(squared_l2_dists)[:10]   # Top-10 smallest

print("\nTop-10 Document IDs by ranking metric:")
print("  Ranked by Cosine Sim:     ", rank_by_cosine)
print("  Ranked by Dot Product:    ", rank_by_dot)
print("  Ranked by Squared L2 Dist:", rank_by_l2)

assert np.array_equal(rank_by_cosine, rank_by_dot), "Cosine and Dot rankings diverge!"
assert np.array_equal(rank_by_cosine, rank_by_l2), "Cosine and L2 rankings diverge!"
print("\n PROOF VERIFIED: All three metrics yield 100% identical rankings on normalized vectors!")


### Visualizing the Relationship: Dot Product vs. Euclidean Distance
Below, we plot the exact relationship between Dot Product and Squared Euclidean Distance for unit-normalized vectors. Notice the perfect straight line with slope $-2$:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 2 - 2(\mathbf{u} \cdot \mathbf{v})$$


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(dot_products, squared_l2_dists, alpha=0.5, color='royalblue', edgecolors='none', s=25)
plt.plot([-1, 1], [4, 0], color='crimson', linestyle='--', linewidth=2, label=r'Theoretical: $d_{L2}^2 = 2(1 - \mathbf{u} \cdot \mathbf{v})$')
plt.title("Empirical Verification: Squared L2 Distance vs. Normalized Dot Product", fontsize=12, pad=10)
plt.xlabel("Dot Product (Cosine Similarity) [Range: -1 to +1]", fontsize=11)
plt.ylabel("Squared L2 Distance [Range: 0 to 4]", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
## 5. Metric Selection Rules for RAG

Choosing the wrong metric for your vector database is one of the most common silent bugs in RAG pipelines. If the embedding model was trained with one objective, querying the index with an incompatible metric will degrade retrieval accuracy.

### Industry Decision Matrix

| Metric | When to use in RAG | Top-K Goal | Compatible Embedding Models | Notes / Gotchas |
| :--- | :--- | :---: | :--- | :--- |
| **Cosine Similarity** | Semantic text retrieval where document length shouldn't bias relevance | **Maximize** score | `text-embedding-3-small/large` (OpenAI), `bge-large-en-v1.5`, `all-MiniLM-L6-v2` | Best default for NLP. Insensitive to document text length variations. |
| **Dot Product (Inner Product)** | Embeddings pre-normalized to unit length **OR** models trained specifically for MIPS | **Maximize** score | Cohere `embed-v3`, DPR (Dense Passage Retrieval), ColBERT late-interaction | Fastest computation. If vectors are unit-normalized, equivalent to Cosine. |
| **Euclidean Distance ($L_2$)** | Physical clustering, visual embeddings, or when vector magnitude carries meaning | **Minimize** distance | Image embeddings (CLIP, ResNet), Audio embeddings, clustering tasks | Sensitive to vector length. If vectors are normalized, equivalent to Cosine. |
| **Manhattan Distance ($L_1$)** | Sparse vectors, count/frequency histograms, or high-dimensional outlier data | **Minimize** distance | Sparse term vectors, SPLADE sparse representations | Rarely used for dense text embeddings. |

---

### The "Metric Mismatch" Pitfall
What happens if you index unnormalized vectors and use Euclidean distance when the model was trained with Cosine loss?
Let's simulate this disaster below:


In [ ]:
# Simulate documents where semantic topic is direction, but length represents text length
# Topic direction: 45-degree angle
topic_direction = np.array([1.0, 1.0]) / np.sqrt(2.0)

# Doc A: Highly relevant short paragraph (same direction, small norm)
doc_a = 1.0 * topic_direction

# Doc B: Highly relevant long article (same direction, large norm)
doc_b = 5.0 * topic_direction

# Doc C: Completely IRRELEVANT document (opposite direction, small norm)
irrelevant_direction = np.array([-1.0, 1.0]) / np.sqrt(2.0)
doc_c = 1.0 * irrelevant_direction

# User Query: Same topic direction
query = 1.2 * topic_direction

# Evaluate using Euclidean Distance (unnormalized)
dist_a = np.linalg.norm(query - doc_a)
dist_b = np.linalg.norm(query - doc_b)
dist_c = np.linalg.norm(query - doc_c)

print("--- UNNORMALIZED EUCLIDEAN DISTANCE ---")
print(f"Dist to Doc A (Relevant short):     {dist_a:.4f}  (Rank 1)")
print(f"Dist to Doc C (IRRELEVANT):         {dist_c:.4f}  (Rank 2)")
print(f"Dist to Doc B (Relevant long):      {dist_b:.4f}  (Rank 3 - FAILED!)")
print(" Disaster: The completely irrelevant document ranked HIGHER than the relevant long document!")

# Evaluate using Cosine Similarity
sim_a = np.dot(query, doc_a) / (np.linalg.norm(query) * np.linalg.norm(doc_a))
sim_b = np.dot(query, doc_b) / (np.linalg.norm(query) * np.linalg.norm(doc_b))
sim_c = np.dot(query, doc_c) / (np.linalg.norm(query) * np.linalg.norm(doc_c))

print("\n--- COSINE SIMILARITY ---")
print(f"Cosine Sim to Doc A (Relevant short): {sim_a:.4f}  (Rank 1)")
print(f"Cosine Sim to Doc B (Relevant long):  {sim_b:.4f}  (Rank 1 tie)")
print(f"Cosine Sim to Doc C (IRRELEVANT):     {sim_c:.4f}  (Rank 3 - Correctly Rejected)")
print(" Cosine similarity evaluated semantic angle accurately regardless of vector length.")


---
## 6. Top-K Search Algorithms from Scratch

Once similarity or distance scores are calculated for all $N$ documents in the database:
$$\mathbf{S} = [s_0, s_1, s_2, \dots, s_{N-1}]$$

The retrieval engine must return the **Top-$K$ items**:
- If using similarity (Cosine/Dot Product): Find indices of the **$K$ largest** scores.
- If using distance ($L_2$/Manhattan): Find indices of the **$K$ smallest** distances.

In production RAG, $K$ is typically small ($K \in [3, 20]$), while $N$ can range from $10^4$ to $10^8$.
How we find these $K$ items dramatically impacts query latency.

We will implement and analyze three algorithmic approaches:
1. **Full Sorting**: $O(N \log N)$
2. **Priority Queue / Bounded Heap**: $O(N \log K)$
3. **QuickSelect (Hoare's Partitioning)**: $O(N)$ expected


### 6.1 Approach 1: Naive Full Sort ($O(N \log N)$)

The most naive way to find the top $K$ items is to sort the entire array of $N$ scores and take the first $K$ elements.

```
Scores: [0.12, 0.95, 0.44, 0.88, 0.03, 0.77]  (N=6, want K=2)
Sorted: [0.95, 0.88, 0.77, 0.44, 0.12, 0.03]  <- Sorts all N elements!
Top-K:  [0.95, 0.88]
```

- **Time Complexity**: $O(N \log N)$
- **Space Complexity**: $O(N)$ extra memory for index array.
- **Flaw**: Sorting all $N$ items is completely wasteful because we don't care about the relative order of the $N - K$ irrelevant items! If $N = 10,000,000$ and $K=10$, sorting all 10 million elements does astronomical amounts of useless work.


In [ ]:
def topk_full_sort(scores: np.ndarray, k: int, largest: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    """Find top-k using complete array sort.
    
    Args:
        scores: 1D array of scores/distances
        k: number of top items to return
        largest: True for similarity (descending), False for distance (ascending)
    Returns:
        (top_indices, top_scores)
    """
    if largest:
        # Sort descending
        sorted_indices = np.argsort(-scores)
    else:
        # Sort ascending
        sorted_indices = np.argsort(scores)
        
    top_indices = sorted_indices[:k]
    top_scores = scores[top_indices]
    return top_indices, top_scores

# Test on toy array
toy_scores = np.array([0.15, 0.89, 0.32, 0.98, 0.45, 0.76, 0.11, 0.65], dtype=np.float32)
top_idx, top_val = topk_full_sort(toy_scores, k=3, largest=True)
print("Toy Scores:   ", toy_scores)
print("Top-3 Indices:", top_idx)
print("Top-3 Scores: ", top_val)


### 6.2 Approach 2: Bounded Priority Queue / Min-Heap ($O(N \log K)$)

Instead of sorting the entire array, we maintain a bounded collection of the best $K$ candidates seen so far using a **Binary Heap**.

#### The Core Mental Model:
To find the **$K$ LARGEST** elements:
- Maintain a **MIN-HEAP** of size $K$.
- Why a *Min-Heap*? Because in a Min-Heap, the root is always the **smallest of the current top-$K$ candidates** (the "gatekeeper").
- As we iterate through each of the $N$ scores:
  1. If heap size $< K$: Push candidate `(score, doc_id)`. Cost: $O(\log K)$.
  2. If heap size $== K$: Compare candidate against the root (`heap[0]`).
     - If `score <= root.score`: Candidate cannot belong in top $K$. Ignore it ($O(1)$ check!).
     - If `score > root.score`: Pop the root and push the candidate (`heapreplace`). Cost: $O(\log K)$.

```
Streaming scores into Min-Heap of size K=3:
Candidate 0.89: [0.89]
Candidate 0.32: [0.32, 0.89]
Candidate 0.98: [0.32, 0.89, 0.98]  <-- Heap is now full (K=3). Root is 0.32.
Candidate 0.45: 0.45 > 0.32! Replace 0.32 -> Heap: [0.45, 0.89, 0.98]
Candidate 0.11: 0.11 < 0.45? Discard immediately! (O(1))
```

- **Time Complexity**: $O(N \log K)$
- **Space Complexity**: $O(K)$ extra memory!
- **When $K \ll N$**: Since $\log K \ll \log N$, this is substantially faster and uses negligible memory.


In [ ]:
class BoundedTopKHeap:
    """Bounded Min-Heap to find top-k maximum scores (similarity search)."""
    
    def __init__(self, k: int):
        self.k = k
        self.heap: List[Tuple[float, int]] = []  # stores (score, doc_id)
        
    def add(self, doc_id: int, score: float):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, (score, doc_id))
        else:
            # Root has the smallest score among top-k candidates
            if score > self.heap[0][0]:
                heapq.heapreplace(self.heap, (score, doc_id))
                
    def get_topk(self) -> List[Tuple[int, float]]:
        """Return sorted top-k in descending order (highest score first)."""
        # Sort the k elements: O(K log K)
        sorted_k = sorted(self.heap, key=lambda x: x[0], reverse=True)
        return [(doc_id, score) for score, doc_id in sorted_k]


class BoundedBottomKHeap:
    """Bounded Max-Heap to find top-k minimum distances (distance search)."""
    
    def __init__(self, k: int):
        self.k = k
        self.heap: List[Tuple[float, int]] = []  # stores (-distance, doc_id)
        
    def add(self, doc_id: int, distance: float):
        # We store negative distance to simulate a Max-Heap with Python's min-heapq
        neg_dist = -distance
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, (neg_dist, doc_id))
        else:
            # Root has the largest distance among current best candidates
            if neg_dist > self.heap[0][0]:  # means distance < -root_neg_dist
                heapq.heapreplace(self.heap, (neg_dist, doc_id))
                
    def get_topk(self) -> List[Tuple[int, float]]:
        """Return sorted top-k in ascending order (smallest distance first)."""
        sorted_k = sorted(self.heap, key=lambda x: x[0], reverse=True)
        return [(doc_id, -neg_dist) for neg_dist, doc_id in sorted_k]


# Verify Top-K Min-Heap on toy scores
heap_tracker = BoundedTopKHeap(k=3)
for doc_id, s in enumerate(toy_scores):
    heap_tracker.add(doc_id, float(s))

heap_results = heap_tracker.get_topk()
print("Top-3 from Bounded Heap [(doc_id, score)]:")
for rank, (doc_id, s) in enumerate(heap_results, 1):
    print(f"  Rank {rank}: Doc ID {doc_id}, Score {s:.4f}")

# Verify exact match with full sort
assert [doc_id for doc_id, _ in heap_results] == list(top_idx), "Heap results do not match full sort!"
print(" Bounded Heap matches Full Sort!")


### 6.3 Approach 3: QuickSelect / Partial Selection ($O(N)$ average)

Can we do even better than $O(N \log K)$?
Yes! In 1961, Sir Tony Hoare invented **QuickSelect** (the selection sibling of QuickSort).

#### The Intuition:
QuickSort sorts both the left and right halves around a pivot.
QuickSelect notices that if we only want the $K$-th largest element, **we only need to recurse into the partition that contains index $K$**!

```
Array: [7, 2, 1, 6, 8, 5, 3, 4]  (want Top-3 largest, pivot=5)
Partition around 5:
[7, 6, 8]  | 5 |  [2, 1, 3, 4]
(3 elements)        (4 elements)

Since left partition has 3 elements, all top-3 are guaranteed to be in the left partition!
We can completely prune and ignore the right partition!
```

- **Time Complexity**:
  - Average: $O(N)$ linear time!
  - Worst Case: $O(N^2)$ (if extremely poor pivots are chosen).
- **Space Complexity**: $O(1)$ auxiliary space (in-place).
- **Caveat**: QuickSelect guarantees that the first $K$ elements are the $K$ largest, but **they are not sorted among themselves**. If the caller needs strictly sorted order, we perform a tiny sort on just those $K$ elements: $O(K \log K)$.
- **In Python/C**: NumPy's famous `np.argpartition` is built on a hybrid QuickSelect algorithm (called *Introselect*)!


In [ ]:
def quickselect_partition(indices: np.ndarray, scores: np.ndarray, low: int, high: int) -> int:
    """Partition helper for QuickSelect (descending order)."""
    pivot_idx = high
    pivot_score = scores[indices[pivot_idx]]
    
    i = low
    for j in range(low, high):
        # We want largest first, so compare scores[indices[j]] >= pivot_score
        if scores[indices[j]] >= pivot_score:
            indices[i], indices[j] = indices[j], indices[i]
            i += 1
            
    indices[i], indices[high] = indices[high], indices[i]
    return i

def quickselect_scratch(indices: np.ndarray, scores: np.ndarray, low: int, high: int, k: int):
    """In-place QuickSelect to place the top-k largest elements in indices[0:k]."""
    if low < high:
        pivot_idx = quickselect_partition(indices, scores, low, high)
        
        if pivot_idx == k:
            return
        elif pivot_idx > k:
            quickselect_scratch(indices, scores, low, pivot_idx - 1, k)
        else:
            quickselect_scratch(indices, scores, pivot_idx + 1, high, k)

def topk_quickselect(scores: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    """Top-k selection using Hoare's QuickSelect algorithm from scratch."""
    N = len(scores)
    indices = np.arange(N)
    
    # In-place partition so that the top-k elements occupy indices[:k]
    quickselect_scratch(indices, scores, 0, N - 1, k)
    
    top_indices = indices[:k]
    # Final sort of only K elements: O(K log K)
    top_indices = top_indices[np.argsort(-scores[top_indices])]
    top_scores = scores[top_indices]
    
    return top_indices, top_scores

# Test QuickSelect on toy scores
qs_idx, qs_val = topk_quickselect(toy_scores.copy(), k=3)
print("QuickSelect Top-3 Indices:", qs_idx)
print("QuickSelect Top-3 Scores: ", qs_val)
assert np.array_equal(qs_idx, top_idx), "QuickSelect failed to match Full Sort!"
print(" QuickSelect matches Full Sort!")


---
## 7. Empirical Benchmarking: Scaling Top-K

How do these selection algorithms behave as the database size $N$ grows from $1,000$ to $500,000$ vectors?
Let's run an empirical benchmark measuring the latency of:
1. **Full Sort** (`np.argsort`)
2. **From-Scratch Min-Heap** (`BoundedTopKHeap`)
3. **NumPy Introselect** (`np.argpartition` - industry standard C-implementation of QuickSelect)


In [ ]:
dataset_sizes = [1000, 5000, 20000, 100000, 300000]
K_target = 10

times_full_sort = []
times_heap = []
times_partition = []

for N_val in dataset_sizes:
    # Generate random similarity scores
    scores_bench = np.random.uniform(low=-1.0, high=1.0, size=N_val).astype(np.float32)
    
    # 1. Benchmark Full Sort
    start = time.perf_counter()
    _ = np.argsort(-scores_bench)[:K_target]
    times_full_sort.append((time.perf_counter() - start) * 1000.0)  # ms
    
    # 2. Benchmark Heap (in Python)
    start = time.perf_counter()
    heap_bench = BoundedTopKHeap(k=K_target)
    for doc_id, s in enumerate(scores_bench):
        heap_bench.add(doc_id, float(s))
    _ = heap_bench.get_topk()
    times_heap.append((time.perf_counter() - start) * 1000.0)  # ms
    
    # 3. Benchmark NumPy's argpartition (Introselect in C)
    start = time.perf_counter()
    part_idx = np.argpartition(-scores_bench, K_target)[:K_target]
    _ = part_idx[np.argsort(-scores_bench[part_idx])]
    times_partition.append((time.perf_counter() - start) * 1000.0)  # ms

print(f"{'N (Vectors)':<12} | {'Full Sort (ms)':<16} | {'Python Heap (ms)':<18} | {'argpartition (ms)':<18}")
print("-" * 72)
for i, n in enumerate(dataset_sizes):
    print(f"{n:<12} | {times_full_sort[i]:<16.4f} | {times_heap[i]:<18.4f} | {times_partition[i]:<18.4f}")


### Visualizing Algorithmic Scaling Curves


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(dataset_sizes, times_full_sort, 'o-', color='crimson', label=r'Full Sort ($O(N \log N)$ - NumPy)')
plt.plot(dataset_sizes, times_heap, 's--', color='darkorange', label=r'Python Min-Heap ($O(N \log K)$ - Python loop)')
plt.plot(dataset_sizes, times_partition, '^-', color='forestgreen', label=r'argpartition / QuickSelect ($O(N)$ - C-optimized)')

plt.title(f"Top-{K_target} Selection Latency vs. Database Size (N)", fontsize=13, pad=12)
plt.xlabel("Number of Vectors (N)", fontsize=11)
plt.ylabel("Execution Time (milliseconds)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
## 8. Industry Reference Implementations (NumPy, SciPy, PyTorch, Faiss)

How do professional vector search libraries and frameworks handle distance computation and Top-K?
Let's inspect how the exact same search is executed using production-grade libraries.

### 1. SciPy (`scipy.spatial.distance.cdist`)
SciPy provides highly optimized C routines for pairwise distances across all metrics.


In [ ]:
from scipy.spatial.distance import cdist

# Create synthetic query (1, d) and corpus (N, d)
q_vec = np.random.randn(1, 64).astype(np.float32)
c_matrix = np.random.randn(500, 64).astype(np.float32)

# Compute pairwise Euclidean distance matrix
scipy_l2 = cdist(q_vec, c_matrix, metric='euclidean').flatten()

# Compute pairwise Cosine distance matrix (1 - cosine_similarity)
scipy_cos = cdist(q_vec, c_matrix, metric='cosine').flatten()

print(f"SciPy computed distances for {len(scipy_l2)} documents.")
print(f"Top-3 nearest by SciPy Euclidean: {np.argsort(scipy_l2)[:3]}")
print(f"Top-3 nearest by SciPy Cosine:    {np.argsort(scipy_cos)[:3]}")


### 2. PyTorch (`torch.topk`)
In GPU-accelerated RAG pipelines, PyTorch computes Top-K directly on VRAM using parallel radix-sort and bitonic selection networks.


In [ ]:
try:
    import torch
    
    # Convert numpy arrays to torch tensors
    t_scores = torch.from_numpy(scores_bench)
    
    # torch.topk runs directly in optimized C++/CUDA
    t_val, t_idx = torch.topk(t_scores, k=K_target, largest=True)
    
    print("PyTorch torch.topk successfully executed!")
    print(f"PyTorch Top-{K_target} indices: {t_idx.numpy()[:5]}...")
except ImportError:
    print("PyTorch not installed in this environment; skipping PyTorch demonstration.")


### 3. Faiss (`faiss.IndexFlatL2` and `faiss.IndexFlatIP`)
**Faiss** (Facebook AI Similarity Search) is the world's gold standard library for dense vector similarity.
- `IndexFlatL2`: Exact brute-force search using squared Euclidean distance.
- `IndexFlatIP`: Exact brute-force search using Inner Product (Dot Product).

Let's test Faiss and verify that its results match our from-scratch implementation bit-for-bit!


In [ ]:
try:
    import faiss
    
    d_dim = 64
    n_docs = 2000
    
    # Create test data (Faiss requires float32 contiguous arrays)
    db_vectors = np.random.randn(n_docs, d_dim).astype(np.float32)
    q_vector = np.random.randn(1, d_dim).astype(np.float32)
    
    # Normalize for Cosine/IP search
    db_vectors_norm = db_vectors / np.linalg.norm(db_vectors, axis=1, keepdims=True)
    q_vector_norm = q_vector / np.linalg.norm(q_vector)
    
    # -------------------------------------------------------------
    # 1. Faiss IndexFlatIP (Inner Product)
    # -------------------------------------------------------------
    index_ip = faiss.IndexFlatIP(d_dim)
    index_ip.add(db_vectors_norm)  # Ingest vectors
    
    k_search = 5
    faiss_scores, faiss_indices = index_ip.search(q_vector_norm, k_search)
    
    # 2. Compare against our Scratch Dot Product + Top-K Heap
    scratch_scores = np.dot(db_vectors_norm, q_vector_norm.flatten())
    scratch_heap = BoundedTopKHeap(k=k_search)
    for doc_idx, s in enumerate(scratch_scores):
        scratch_heap.add(doc_idx, float(s))
    scratch_topk = scratch_heap.get_topk()
    
    print("Faiss IndexFlatIP Top-5:")
    print("  Indices: ", faiss_indices[0])
    print("  Scores:  ", np.round(faiss_scores[0], 4))
    
    print("\nOur Scratch Min-Heap Top-5:")
    print("  Indices: ", [idx for idx, _ in scratch_topk])
    print("  Scores:  ", [round(s, 4) for _, s in scratch_topk])
    
    assert list(faiss_indices[0]) == [idx for idx, _ in scratch_topk], "Faiss indices diverge from scratch!"
    print("\n 100% BIT-FOR-BIT MATCH WITH FAISS!")

except ImportError:
    print("Faiss not installed in this environment; skipping Faiss demonstration.")
    print("You can install it via: pip install faiss-cpu")


---
## 9. Pros, Cons & Decision Tradeoffs

### Comparison of Distance & Similarity Metrics

| Metric | Mathematical Formula | Pros | Cons | Best Used When... |
| :--- | :--- | :--- | :--- | :--- |
| **Cosine Similarity** | $\frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$ | Length-invariant; matches human semantic judgment; standard for NLP. | Requires vector normalization; slightly more computation if unnormalized. | Default choice for RAG text embeddings (OpenAI, HuggingFace, BGE). |
| **Dot Product (Inner Product)** | $\sum u_i v_i$ | Fastest computation; hardware accelerated by BLAS/GPU matrix multiply. | Sensitive to vector length; unbounded scores ($-\infty$ to $+\infty$). | Vectors are pre-normalized, or the model uses length to encode document popularity. |
| **Euclidean Distance ($L_2$)** | $\sqrt{\sum (u_i - v_i)^2}$ | Intuitive physical distance; satisfies all metric triangle inequalities. | Suffering from the Curse of Dimensionality in high dimensions; length-dependent. | Image/audio retrieval, geometric clustering, or when vectors are already normalized. |
| **Manhattan Distance ($L_1$)** | $\sum \|u_i - v_i\|$ | Less sensitive to dimensional outliers; well-behaved on sparse feature vectors. | Non-differentiable at origin; rarely aligned with dense language representations. | Sparse lexical representations (SPLADE, BM25 count vectors). |

---

### Comparison of Top-K Search Algorithms

| Algorithm | Time Complexity | Extra Space | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **Full Sort (`argsort`)** | $O(N \log N)$ | $O(N)$ | Simple to write; yields completely ordered ranking of all items. | Completely unacceptable for large $N$; sorts millions of irrelevant items. |
| **Bounded Min-Heap** | $O(N \log K)$ | $O(K)$ | Ideal for streaming data; tiny memory footprint; orders of magnitude faster when $K \ll N$. | Python overhead in pure loops; needs C/C++ acceleration to beat vectorized BLAS. |
| **QuickSelect (`argpartition`)** | $O(N)$ average | $O(1)$ | Optimal theoretical complexity; blazing fast in C; standard in modern vector engines. | Unordered top-$K$ (requires small $O(K \log K)$ sort at the end); worst case $O(N^2)$. |


---
## 10. Summary & What's Next

### Key Takeaways from Notebook 01:
1. **Vectors in $\mathbb{R}^d$**: Embeddings represent concepts as points in high-dimensional vector spaces ($d=384$ to $3072$).
2. **The Big Four Metrics**: Euclidean ($L_2$), Manhattan ($L_1$), Dot Product, and Cosine Similarity.
3. **The Core Equivalence**: For unit-normalized vectors ($\|\mathbf{x}\|_2 = 1$):
   $$\text{Cosine}(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v} = 1 - \frac{1}{2} \|\mathbf{u} - \mathbf{v}\|_2^2$$
   Always normalize your vectors offline so your production vector database can replace slow square roots and divisions with a single, blazing-fast dot product!
4. **Top-K Selection**: Full sorting ($O(N \log N)$) is a scalability trap. Use Bounded Priority Queues ($O(N \log K)$) for streaming or QuickSelect / `argpartition` ($O(N)$) for batch scoring.

---

### In the Next Notebook:
👉 **Notebook 02: Exact Brute-Force (Flat) Search & Baseline Benchmarking**
- Building an industrial-strength exact k-NN retriever from scratch.
- Vectorized batch matrix multiplication ($Q \cdot D^T$).
- Memory footprint vs. query latency scaling curves up to $1,000,000$ vectors.
- Why exact search hits the "Hardware Wall" (RAM bandwidth & CPU limits).
- Establishing the exact **Ground Truth** benchmark required to evaluate Approximate Nearest Neighbor (ANN) indexes!
